# LionAG2: Recursive Exploratory Research with AG2 beta — structured output with response_schema (2/10)

In this tutorial, we will take a look at how AG2 handles structured output and how we can use that in our research pipeline.

**Structured output** is when an AI model produces output that can be parsed and validated into a data object, enabling AI output to be manipulated programmatically. This forms the basis of the bulk of today's workflow automation and processes — in fact, the tool use ability of language models also roots in structured output. Consider the Exa research example from yesterday, how exactly did the tool get triggered?

We asked the model about something, with explicit mandate on Exa tool usage, and the following things happened:

1. The tool schema (what to put into the tool interface as parameters) gets injected into the instruction sent to the model alongside your user prompt
2. The model creates structured output matching function signatures of those tools
3. The function gets matched to a real function object, and arguments get parsed, tools invoked, and results stored
4. The framework sends a second API call, including the tool results, the model then returns final outputs

Today we will extend that with one more step — we will ask the model to also produce a structured output in its final response. The point of which will be obvious in the next tutorial.

## Setup

```bash
pip install 'ag2[openai, exa]'
```

Make sure you have OpenAI and Exa API keys saved in your environment.

In [1]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.tools import ExaToolkit

load_dotenv()

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)
exa_tool = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

## Structured research findings

### 1. Define structures in Pydantic models

In [2]:
class Citation(BaseModel):
    title: str = Field(description="The cited work or source.")
    relevance: str = Field(description="One sentence on why this source supports the finding.")


class Finding(BaseModel):
    """One self-contained research result."""
    topic: str = Field(description="The specific aspect of the question this finding addresses.")
    summary: str = Field(description="2-3 sentences capturing the core mechanism or claim.")
    citations: list[Citation] = Field(description="Sources behind the summary; emit [] if none.")
    novelty: float = Field(ge=0.0, le=1.0, description="0 = textbook, 1 = cutting edge.")

### 2. Declare the agent with `response_schema`

In [3]:
theorist = Agent(
    name="theorist",
    prompt="You're a careful theoretical physicist. Be specific.",
    config=config,
    response_schema=Finding,
    tools=[exa_tool],
)

### 3. Run and collect the result

In [4]:
reply = await theorist.ask(
    "Drill into the frontier of high-Tc superconductivity."
)
finding: Finding = await reply.content()

print(type(finding).__name__, "|", finding.topic, "| novelty=", finding.novelty)

Finding | Frontier directions in high-Tc superconductivity | novelty= 1.0


Since `finding` is now a Pydantic model object, we can directly use its attributes like `finding.summary` and others.

In [5]:
lines = [f"### {finding.topic} \u2014 novelty={finding.novelty:.2f}", "", finding.summary]
for c in finding.citations:
    lines.append(f"- **{c.title}** \u2014 {c.relevance}")
display(Markdown("\n".join(lines)))

### Frontier directions in high-Tc superconductivity — novelty=1.00

The frontier is split between discovering new platforms and resolving the microscopic pairing mechanism in established ones. Cuprates remain the benchmark, with recent work focusing on direct CuO2-plane probes in infinite-layer compounds; nickelates have emerged as the most important new oxide family, especially the pressurized bilayer La3Ni2O7 and related multilayers. In parallel, moiré superconductors and compressed hydrides are pushing the field toward unconventional pairing regimes and higher critical temperatures, but synthesis, pressure conditions, and microscopic characterization remain the main bottlenecks.
- **Unveiling high-Tc superconductivity: probing CuO2 planes in infinite-layer cuprates** — Open-access 2025 review emphasizing direct CuO2-plane measurements and the role of infinite-layer cuprates in resolving the mechanism.
- **Recent progress in nickelate superconductors** — 2025 review summarizing the latest nickelate systems, including LaNiO2 and La3Ni2O7, and the key open issues.
- **Twisted Nodal Superconductors** — 2025 arXiv review highlighting moiré superconductivity as a new frontier with open questions about topology and time-reversal symmetry breaking.
- **When superconductivity crosses over: From BCS to BEC** — 2024 Reviews of Modern Physics article framing modern high-Tc-adjacent systems in the BCS-BEC crossover and artificial-material platforms.

## Notes

In this setup, we are asking the model to produce the citation, which is typically not the most desirable practice — models can output whatever they wish without regard for reality or what is actually in their context. For example, how do you know that the model actually used certain info from the cited sources? If so, where? How do you know that the model didn't hallucinate the citation altogether?

This is the reason why we like structured output — it enables these kinds of questions to be answered **programmatically**, instead of dumping everything to another model and asking:

> hey dude, is this free of hallucination and all citations real?

In the next tutorial, we will explore how to make use of the structured output from one agent run, and then see how to wire multiple runs together.

For more info on structured output in AG2, including advanced features like [callable `@response_schema`](https://docs.ag2.ai/latest/docs/beta/structured_output/#custom-validation-with-response_schema) for custom validation and side effects, read the [docs](https://docs.ag2.ai/latest/docs/beta/structured_output/).

## Up next

Day 3: **wiring agents together**.

One agent returning a typed object is useful. But what if we could take that `Finding`'s open questions and hand the most novel one to a second agent that produces a testable hypothesis? That's the multi-agent handoff pattern, and structured output is what makes it trivial.